# Detecting Objects

You may have already used the Face Detector to detect faces in a scene. The **Object Detector** is more general — a computer vision model trained to spot dozens of different everyday things at the same time.

Codetto uses **EfficientDet-Lite0**, a small model trained on a picture collection called **COCO**. It can recognize **80 categories**: people, chairs, laptops, phones, cups, books, cars, bikes, dogs, cats, and many more.

Each object found will come back as a **bounding box** with a **confidence** score, and a **label** telling you *what* it is.

Run the cell below and click **Allow**. Point your camera around the room for about eight seconds and watch the boxes appear.

In [ ]:
from codetto import cv, graphics
import time

canvas = graphics.canvas()
camera = cv.start_camera(canvas)
detector = cv.start_object_detector(camera)

try:
  end = time.time() + 8
  while time.time() < end:
    objects = detector.get_detections()
    canvas.draw_bounding_boxes(objects)
finally:
  detector.stop()
  camera.stop()

# The Detector Object

The three lines of code above setup the object detector:

- `cv.start_camera(canvas)` turns the webcam on.
- `cv.start_object_detector(camera)` starts the object model and hands you back a **detector object**.
- `detector.get_detections()` gives back a list of everything it can see right now.

The bounding boxes are drawn each time with `canvas.draw_bounding_boxes(...)`.

# What a Detection Contains

Each object in the list is a **dictionary**. Coordinates, width, and a height provide a bounding box around the object, plus a `type`:

| Field | Meaning |
|---|---|
| `type` | The label, e.g. `"cup"` or `"laptop"` |
| `x`, `y` | Top-left corner of the box, in pixels |
| `w`, `h` | Width and height of the box, in pixels |
| `confidence` | How sure the model is, from `0.0` to `1.0` |

The cell below prints those fields once every 240 frames. Point the camera at different things and switch to the console to watch the labels and numbers change. Press **Stop** when you are done.

In [ ]:
from codetto import cv, graphics

canvas = graphics.canvas()
camera = cv.start_camera(canvas)
detector = cv.start_object_detector(camera)

frame = 0
try:
  while True:
    objects = detector.get_detections()
    canvas.draw_bounding_boxes(objects)
    if frame % 240 == 0:
      print(f'--- {len(objects)} object(s) ---')
      for obj in objects:
        print(f"  {obj['type']}: x={obj['x']} y={obj['y']} "
            f"size={obj['w']}x{obj['h']} confidence={obj['confidence']:.0%}")
    frame += 1
finally:
  detector.stop()
  camera.stop()

# Confidence Scores

Every object comes with a `confidence` score, and you can use a **threshold** to keep only the detections the model is sure enough about.

- A **low** threshold shows more objects, but some labels will be wrong.
- A **high** threshold shows fewer objects, only the confident ones.

Move the slider and restart the cell to test new values. Try a low value and see what strange things it labels. Press **Stop** when you are done.

In [ ]:
from codetto import cv, graphics

MIN_CONFIDENCE = 0.5 #@param {type:"slider", min:0.1, max:1.0, step:0.05}

canvas = graphics.canvas()
camera = cv.start_camera(canvas)
detector = cv.start_object_detector(camera)

try:
  while True:
    objects = detector.get_detections()
    sure = [obj for obj in objects if obj['confidence'] >= MIN_CONFIDENCE]
    canvas.draw_bounding_boxes(sure)
finally:
  detector.stop()
  camera.stop()

# Keeping a List of What You've Seen

The `type` field is just a string, so you can collect the labels as the camera runs. A **set** is a container that keeps each value only once, which makes it perfect for a "things I have spotted" list.

The cell below adds every new label to a set and prints it the first time it appears. Walk the camera around the room and see how long a list you can build. Press **Stop** when you are done.

In [ ]:
from codetto import cv, graphics

canvas = graphics.canvas()
camera = cv.start_camera(canvas)
detector = cv.start_object_detector(camera)

seen = set()
try:
  while True:
    objects = detector.get_detections()
    canvas.draw_bounding_boxes(objects)
    for obj in objects:
      if obj['type'] not in seen:
        seen.add(obj['type'])
        print('New object spotted:', obj['type'], ' | total so far:', len(seen))
finally:
  detector.stop()
  camera.stop()

# Check Your Understanding